# Baseline Model

## Package Import & Path Settings

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import hashlib
from pathlib import Path
from collections import defaultdict

from PIL import Image, UnidentifiedImageError

# pytorch
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from torchvision import transforms
from torch import nn
import torch

# data split
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

# preprocessing & structure
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer

# baseline models
from sklearn.dummy import DummyRegressor
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, log_loss

# others
from concurrent.futures import ThreadPoolExecutor
from src.dataset import load_filtered_imagefolder, rgba_to_rgb_with_bg
from tqdm.auto import tqdm

# path settings
from src import MODELS_DIR, PARAMS_PATH, SEED, PET_IMAGES_DIR, get_device
from src.utils import plot_hist
DEVICE = get_device()
print(f'Device: {DEVICE}')
print(f'MODELS_DIR: {MODELS_DIR}')
print(f'PARAMS_PATH: {PARAMS_PATH}')
print(f'SEED: {SEED}')
print(f'PET_IMAGES_DIR: {PET_IMAGES_DIR}')

# project constants
label = {0:'cat', 1:'dog'}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
SAMPLE_SIZE = 256


Device: cuda
MODELS_DIR: D:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\outputs\models
PARAMS_PATH: D:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\outputs\params
SEED: 37
PET_IMAGES_DIR: D:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\data\raw\PetImages


## Helper Functions

## Data Loading


In [2]:
train_tf = transforms.Compose([
    transforms.Resize(SAMPLE_SIZE),
    transforms.RandomCrop(SAMPLE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
val_tf = transforms.Compose([
    transforms.Resize(SAMPLE_SIZE),
    transforms.CenterCrop(SAMPLE_SIZE),    
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [4]:
bad_path = PET_IMAGES_DIR / 'bad_files.json'
dup_group_path = PET_IMAGES_DIR / 'duplicate_groups_exact.json'
ignore_path = PET_IMAGES_DIR / 'ignore_files_list.json'

ds_aug, excl_set = load_filtered_imagefolder(PET_IMAGES_DIR, bad_path, dup_group_path, ignore_path, train_tf, rgba_to_rgb_with_bg) # the dataset with augmentation
ds, _ = load_filtered_imagefolder(PET_IMAGES_DIR, bad_path, dup_group_path, ignore_path, val_tf, rgba_to_rgb_with_bg) # the dataset without any augmentation

print(f'Dataset Loaded from {PET_IMAGES_DIR}')
print(f'Number of valid samples: {len(ds)}')
print(f'Number of samples excluded: {len(excl_set)}')

Dataset Loaded from D:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\data\raw\PetImages
Number of valid samples: 24968
Number of samples excluded: 32


### Test Data Split

Train / Test Data Split: 50%

In [5]:
ds.samples = sorted(ds.samples, key=lambda x: x[0])
ds.targets = [l for _, l in ds.samples]
ds_path = [p for p, _ in ds.samples]
X_train_path, X_test_path, y_train, y_test = train_test_split(ds_path, ds.targets, test_size=0.5, random_state=SEED, stratify=ds.targets)
path_dict = {p: i for i, p in enumerate(ds_path)}
idx_tr = np.array([path_dict[p] for p in X_train_path])
idx_test = [path_dict[p] for p in X_test_path]
ds_test = Subset(ds, idx_test)

In [6]:
len(idx_tr)

12484

### Cross Validation

In [7]:
n_splits = 10
test_size = 0.1
skf = StratifiedShuffleSplit(n_splits=10, test_size=0.1, random_state=SEED)

## Baseline Model

A simple CNN model is constructed with PyTorch. The model consists of the following components:
1. **Feature Extraction**:
   1. Conv2D

In [8]:
class CNN_Clf(nn.Module):
    def __init__(self, num_class=2):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(64, num_class)

        pass   

    def forward(self, x):
        x = self.feature(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)
        return logits
    
    @ torch.no_grad()
    def predict(self, x):
        self.eval()
        return self.forward(x)
    
    @ torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.softmax(self.forward(x), dim=1, dtype=torch.float32)
    
    @ torch.no_grad()
    def predict_id(self, x):
        self.eval()
        proba = self.predict_proba(x)
        ids = torch.argmax(proba, axis=1)       
        return ids


Smoke test result shows no shape/data type error during the inferencing.

In [9]:
a = np.random.rand(32, 3, 128, 128).astype('float32')
a_tr = torch.from_numpy(a).to(DEVICE)
cnn_clf = CNN_Clf(2).to(DEVICE)
logits = cnn_clf.predict(a_tr).to('cpu').detach().numpy()
print(logits[0:5])


[[-0.08062814  0.0848016 ]
 [-0.08042483  0.08602983]
 [-0.08057436  0.08575941]
 [-0.08034916  0.08516582]
 [-0.080136    0.08554284]]


### Training Loop

In [56]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(idx_tr, y_train)
proba = dummy_model.predict_proba(idx_tr)
dummy_loss = log_loss(y_train, proba, labels=[0,1])

In [10]:
loss_fn = torch.nn.CrossEntropyLoss()

epochs = 10
batch_size = 32

hist = []
fold_pbar = tqdm(skf.split(idx_tr, y_train), total=skf.get_n_splits(), desc="CV folds")

for fold, (tr_idx, val_idx) in enumerate(skf.split(idx_tr, y_train)):
    ds_fold_tr = Subset(ds_aug, idx_tr[tr_idx].tolist())
    ds_fold_val = Subset(ds, idx_tr[val_idx].tolist())
    dl_fold_tr = DataLoader(ds_fold_tr, batch_size, shuffle=True)
    dl_fold_val = DataLoader(ds_fold_val, batch_size, shuffle=False)
    fold_hist = {'loss':[], 'val_loss':[]}
    cnn_clf = CNN_Clf(2).to(DEVICE)
    optim = torch.optim.Adam(cnn_clf.parameters())       
    for e in tqdm(range(epochs), desc=f"Fold {fold} training", leave=False):
        fold_tr_size = 0
        fold_val_size = 0
        total_loss = 0 
        total_val_loss = 0
        cnn_clf.train()
        for x, y in dl_fold_tr:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            optim.zero_grad()
            logits = cnn_clf(x)            
            loss = loss_fn(logits, y.long())
            bs = x.size(0)
            fold_tr_size += bs
            total_loss += loss.item() * bs
            train_loss = total_loss / fold_tr_size
            loss.backward()
            optim.step()
        fold_hist['loss'].append(train_loss)
        # validation
        cnn_clf.eval()
        with torch.no_grad():
            for x, y in dl_fold_val:
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                logits = cnn_clf(x)
                loss = loss_fn(logits, y.long())
                bs = x.size(0)
                fold_val_size += bs
                total_val_loss += loss.item() * bs
                val_loss = total_val_loss / fold_val_size
        fold_hist['val_loss'].append(val_loss)
    fold_pbar.set_postfix(
                            fold=fold,
                            train_loss=f"{train_loss:.4e}",
                            val_loss=f"{val_loss:.4e}",
                            refresh=True
                        )
    
    hist.append(fold_hist)
    


CV folds:   0%|          | 0/10 [00:00<?, ?it/s]

Fold 0 training:   0%|          | 0/10 [00:00<?, ?it/s]

AttributeError: 'str' object has no attribute 'mode'

In [ ]:

fig, ax = plot_hist(hist)


ax[0].axhline(y=dummy_loss, color='red', linestyle='--', label=f'Dummy Loss: {dummy_loss:.4f}')
ax[1].axhline(y=dummy_loss, color='red', linestyle='--', label=f'Dummy Loss: {dummy_loss:.4f}')
ax[0].set_yscale('log')
ax[1].set_yscale('log')
ax[0].legend()
ax[1].legend()
        